# Mevzuat RAG: cevap üretimi (Colab)

Arama bilgisayarda yapılır; bu defter yalnız dil modelini çalıştırır (K-005).

1. **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU.**
2. Hücreleri sırayla çalıştır. İkinci hücre istem paketini ister: `results/istem/<ad>.jsonl` (bilgisayarda `scripts/istem_paketi.py` üretir). Pilot paketi (3 + 2 kaynak): `dev-e5-demo.jsonl`. Demonun güncel paketi (5 + 2): `dev-e5-demo-b5w2.jsonl`.
   Pilotta defteri iki kez çalıştır: önce `Qwen/Qwen3-4B-Instruct-2507`, sonra `Qwen/Qwen3-8B` (ikinci hücredeki `MODEL`).
3. Son hücre cevap dosyasını indirir. İndirileni bilgisayarda `results/cevap/ham/` klasörüne koy. Bilgisayarda: `uv run python scripts/cevap_ozet.py results/istem/<ad>.jsonl <indirilen>.jsonl`

Gizli anahtar gerekmez: modeller herkese açık, Apache-2.0. Üretim açgözlü (sıcaklık 0); aynı paket ve aynı model aynı cevabı vermeli. Model adı, revizyonu ve kütüphane sürümleri cevap dosyasının ilk satırına yazılır.

In [ ]:
!pip -q install "transformers>=4.56" accelerate bitsandbytes

In [ ]:
# Pilot adayları (K-005). 4B T4'e 16 bit sığar; 8B 16 bitte sığmaz, 4 bit yüklenir.
MODEL = "Qwen/Qwen3-4B-Instruct-2507"  # ya da "Qwen/Qwen3-8B"
DORT_BIT = MODEL == "Qwen/Qwen3-8B"  # 4B için de True yapılabilir (anlamsız çıktı uyarısı gelirse)
MAX_YENI = 400  # cevap başına en fazla üretilen token
TOPLU = 4       # aynı anda üretilen soru; uzun istemler (~13 000 karakter) var, bellek yetmezse 2

from google.colab import files
yuklenen = files.upload()
PAKET = next(iter(yuklenen))
print(PAKET)

In [ ]:
import json, time, torch, transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

assert torch.cuda.is_available(), "GPU yok: Çalışma zamanı türünü T4 GPU yap"
print(torch.cuda.get_device_name(0))
tok = AutoTokenizer.from_pretrained(MODEL)
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
nicel = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16) if DORT_BIT else None
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float16, device_map="cuda",
                                             quantization_config=nicel).eval()

In [ ]:
satirlar = [json.loads(s) for s in open(PAKET, encoding="utf-8") if s.strip()]
# kısa istemler bir arada: dolgu azalır, uzunlar aynı gruba yığılıp belleği taşırmaz
satirlar.sort(key=lambda s: sum(len(m["content"]) for m in s["mesajlar"]))

def metne(mesajlar):
    # Qwen3-8B düşünme modunu destekliyor; kapalı, yalnız cevap
    ek = {"enable_thinking": False} if MODEL == "Qwen/Qwen3-8B" else {}
    return tok.apply_chat_template(mesajlar, tokenize=False, add_generation_prompt=True, **ek)

cikti, t0 = [], time.time()
for i in range(0, len(satirlar), TOPLU):
    grup = satirlar[i:i + TOPLU]
    girdi = tok([metne(s["mesajlar"]) for s in grup], return_tensors="pt", padding=True).to("cuda")
    with torch.no_grad():
        uretilen = model.generate(**girdi, max_new_tokens=MAX_YENI, do_sample=False,
                                  temperature=None, top_p=None, top_k=None,
                                  pad_token_id=tok.pad_token_id)
    yeni = uretilen[:, girdi["input_ids"].shape[1]:]
    for s, g, n in zip(grup, yeni, girdi["attention_mask"].sum(dim=1)):
        cikti.append({"id": s["id"], "cevap": tok.decode(g, skip_special_tokens=True).strip(),
                      "girdi_token": int(n), "cikti_token": int((g != tok.pad_token_id).sum())})
    print(f"{min(i + TOPLU, len(satirlar))}/{len(satirlar)}  {time.time() - t0:.0f} s")

# T4 bf16 desteklemez, model fp16 yüklenir; bazı modellerde fp16 taşması boş ya da "!!!!" çıktı verir
bozuk = [c["id"] for c in cikti if not c["cevap"].strip() or c["cevap"].count("!") > len(c["cevap"]) / 2]
print(f"boş ya da anlamsız cevap: {len(bozuk)} / {len(cikti)}", bozuk[:10])
if len(bozuk) > len(cikti) // 10:
    print("UYARI: fp16 taşması olabilir. İkinci hücrede DORT_BIT = True yapıp yeniden çalıştır.")


In [ ]:
ust = {"model": MODEL, "revizyon": getattr(model.config, "_commit_hash", None), "dort_bit": DORT_BIT,
       "max_yeni": MAX_YENI, "toplu": TOPLU, "paket": PAKET, "gpu": torch.cuda.get_device_name(0),
       "torch": torch.__version__, "transformers": transformers.__version__,
       "sure_s": round(time.time() - t0), "zaman": time.strftime("%Y-%m-%dT%H:%M:%S")}
ad = PAKET.removesuffix(".jsonl") + "__" + MODEL.split("/")[-1] + ".jsonl"
with open(ad, "w", encoding="utf-8") as f:
    f.write(json.dumps({"_ust": ust}, ensure_ascii=False) + "\n")
    for c in cikti:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")
print(ust)
files.download(ad)